# 04. 시간 경과와 날짜별 이력

- **theta**: 시장 그대로 평가일만 이동 (`Market.roll`, ForwardForward: 오늘의 선도가 실현).
- **이력**: 날짜마다 시장을 만들고 **결과 숫자만 표로** 남긴다.

> 데이터: `tests/fixtures/market_sample.txt`는 **합성 샘플**(`날짜 키 값`)이다. 흐름을 익히는 용도이고 숫자로 시장을 해석하지 않는다.

In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd
import QuantLib as ql

from quanttool.config import load_config
from quanttool.market import build_market
from quanttool.market.quotes import read_quotes

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
cfg = load_config(ROOT / "configs")
quotes = read_quotes(ROOT / "tests" / "fixtures" / "market_sample.txt")  # 합성 샘플
ASOF = date(2026, 10, 23)

from quanttool.analysis.risk import theta
from quanttool.market.curves import curve_shape, curve_table
from quanttool.portfolio import Portfolio

market = build_market(ASOF, quotes[ASOF], cfg)
book = Portfolio.from_file(ROOT / "positions" / "ust_book.yaml", products=cfg.products).build(
    market, cfg
)

## 1. theta: 날짜를 옮기면 커브는 어떻게 되나

**커브를 얼린다**: `build_market`이 만든 커브는 기준일이 asof(10/23)로 못박혀 있고, 시세 Quote가 바뀌어도 다시 부트스트랩하지 않는다 (`linked=True`만 예외). 기준일이 전역 평가일을 따라가게 두면, 평가일만 바꿔도 "10/23 시세로 10/26 결제일 기준 부트스트랩"한 섞인 커브가 생긴다. 얼려 두면 "10/23 시장"이 정확히 하나다.

**오늘 커브를 다른 날 그대로 쓰지 않는다**: `market.roll(d)`가 시장 객체를 함께 옮긴다. 가정은 **ForwardForward**, 즉 오늘 커브의 d 시점 선도 커브가 d의 spot 커브가 된다.

| 객체 | roll(d) |
|---|---|
| 커브 | 기준 커브 → `ImpliedTermStructure(오늘 커브, d)`: DF_d(t) = DF(d+t) / DF(d). 요인 층은 그 위에 그대로 |
| vol 서페이스 | 선도 분산이 실현: σ'²(T)·(T−t_d) = σ²(T)·T − σ²(d)·t_d |
| 지수 fixing | [asof, d) 사이 fixing = 오늘의 예측 선도 |
| Heston | 굴린 서페이스에 다시 캘리브레이션 |
| 현물 | 그대로 |

`theta(book, market, d)`는 roll(d) → 재평가 → roll(asof)이다. 이 가정에서는 "시장이 오늘 가격에 담긴 기대대로 흘러간다". 그래서 같은 커브로 가격 낸 포지션의 theta ≈ 하루치 단기금리 carry이고 **roll-down은 빠진다**. roll-down을 보려면 "만기별 제로금리 그대로" 가정이 필요하다 (`CurveFactors.roll`에 방식 하나 추가하는 자리).

In [ ]:
th = theta(book, market, date(2026, 10, 26))  # 다음 영업일 (주말 포함 3일)
carry = book.npvs() * (1 / market["UST"].discount(3 / 365) - 1)  # NPV × 3일 단기금리 (대략)
pd.DataFrame({"theta": th, "NPV × 3일 단기 carry": carry}).round(2)

## 2. 날짜별 커브 이력

In [ ]:
zero_hist, shape_hist = {}, {}
for d, q in quotes.items():
    m = build_market(d, q, cfg)
    with m.active():
        table = curve_table(m["UST"], cfg.conventions["UST"])
    zero_hist[d] = table["zero_cc"] * 100
    shape_hist[d] = curve_shape(table)
display(pd.DataFrame(zero_hist).T.round(4))
pd.DataFrame(shape_hist).T

## 3. 평가일이 바꾸는 것, 그리고 `active()`
QuantLib에는 **전역 평가일**(`Settings.evaluationDate`)이 하나 있다. 결제일, 경과이자, "이 현금흐름은 이미 지났나", "이 fixing은 과거인가"가 모두 이 값으로 정해진다. `with market.active():`는 이 값을 잠시 시장의 날짜(`market.date`, roll하면 d)로 두고 끝나면 되돌린다. `Portfolio.npvs()`는 안에서 알아서 쓴다.

아래 표는 일부러 **어색한 경우**를 함께 보여준다.
- `평가일만 2/16`: `evaluation_date`로 전역 평가일만 옮긴다. 결제일·경과이자·clean은 2/16 기준이 되는데, NPV는 커브 기준일(10/23)로 할인하므로 그대로다(2/15 쿠폰도 아직 포함) — 날짜가 섞인 값이다.
- `roll(2/16)`: 시장도 2/16으로 옮긴다(ForwardForward). 이것이 "2/16의 시장에서 본 값"이다.

평가일과 시장 날짜를 섞지 않으려고 `active()`와 `roll`이 있다.

In [ ]:
from quanttool.market.curves import evaluation_date

bond = book.instruments["UST_10Y"]
d = date(2027, 2, 16)


def row(label):
    return {
        "경우": label,
        "평가일": ql.Settings.instance().evaluationDate.ISO(),
        "결제일": bond.settlementDate().ISO(),
        "경과이자": bond.accruedAmount(),
        "clean": bond.cleanPrice(),
        "NPV": bond.NPV(),
    }


rows = []
with market.active():
    rows.append(row("10/23 시장 (active)"))
with evaluation_date(d):
    rows.append(row("평가일만 2/16 (커브는 10/23)"))
market.roll(d)
with market.active():
    rows.append(row("roll(2/16): 시장도 2/16"))
market.roll(ASOF)
pd.DataFrame(rows).set_index("경우").round(4)